### 프롬프트 템플릿 Agent tool rag에서 표준
- role : LLM이 어떤 역할을 할지 정함
- instruction : 답변 규칙과 형식을 정리
- context : 데이터베이스 검색결과처럼 답변에 참고할 실제 정보
- query : 실제 질문


In [40]:
# 사용자 질문
# 적절한 tool 호출 -> 라우터 ( Fast API )
# 수정한 정보를 -> LLM 전달
# 최종답변은 LLM

In [41]:
import os
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv(override=True)

api_key = os.getenv('OPENAI_API_KEY')
if not api_key:
    raise EnvironmentError('openai api key .....')

class OpenAILLM:
    def __init__(self,model:str = 'gpt-5.4-nano'):
        self.client = OpenAI(api_key=api_key)
        self.model = model
    def generate(self, prompt:str) -> str:
        response = self.client.chat.completions.create(
            model = self.model,
            messages =[
                {"role":"system", "content":"You are an ecomerce recommendation assistant, Return only valid JSON"},
                {"role":"user", "content":prompt}
            ],
            temperature=0,
            response_format={'type':'json_object'}
        )
        return response.choices[0].message.content
llm = OpenAILLM()
llm.model  

'gpt-5.4-nano'

In [42]:
# 데이터
do_search_results = [
    {"id":"p1", "name":"고성능 노트북","category":"가전"},
    {"id":"p2", "name":"사무용 랩탑","category":"가전"},
    {"id":"p3", "name":"미러리스 카메라","category":"가전"},
]
context_string = ""
for item in do_search_results:
    context_string += f"- 상품명:{item['name']} | 카테고리:{item['category']}"

In [43]:
from  textwrap import dedent  #  들여쓰기 indent를 제거
import json
# 프롬프트 템플릿 - 고정된 구조
# 프롬프트생성 + llm호출 + 파싱
def recommend_product(user_question:str, context:str) ->dict:    
    prompt = dedent(f'''
                        당신은  사용자의 질문에 정확히 응답하는 ai 시스템 입니다.
                        사용자의 질문과 context를 보고 질문의 의도에 맞게 출력하세요

                        [컨텍스트]
                        {context}

                        [질문]
                        {user_question}                        
                        
                        [출력]
                        답변은 반드시 아래와 같은 json형태로 
                        {
                            {
                                "assistant" : "출력내용",
                                "reason":"사유"                            
                            }
                        }

                        ''')
    response = llm.generate(prompt)
    data = json.loads(response)
    return json.dumps(data, indent=2, ensure_ascii=False)
    
    

## LLM 응답

In [44]:
import json
response = llm.generate(raw_prompt)
data = json.loads(response)
print(json.dumps(data, indent=2, ensure_ascii=False))

{
  "recommended_product": "고성능 노트북",
  "reason": "코딩 작업에 필요한 성능(빠른 처리와 원활한 멀티태스킹)을 기대할 수 있는 고성능 노트북을 추천합니다. 가전 카테고리 내에서 신뢰도 높은 선택지로, 개발 환경(IDE, 로컬 서버, 빌드/테스트) 사용에 적합합니다."
}


## 라우팅
- 들어온 질문을 보고 어느 경로로 보낼지 결정하는 단계

In [45]:
def route_qeustion(question:str)->str:
    lower_question = question.lower()
    if any(keyword in lower_question for keyword in ['뉴스','기사','검색','찾아줘','최신','오늘']):
        return "news_tool"
    elif any(keyword in lower_question for keyword in ['계산','더하기','곱하기','합계','몇','얼마']):
        return "calculator_tool"
    elif any(keyword in lower_question for keyword in ['기억','기록','선호','메모','이전']):
        return "memory_tool"
    elif any(keyword in lower_question for keyword in ['추천','골라','비교']):
        return "llm_recommendation"
    else:
        return "general_llm"
sample_questions = [
    "3개 상품을 2개씩 주문하면 총 몇 개인가?",
    "나는 코딩용 노트북을 좋아한다는 점을 기억해줘.",
    "AI 에이전트 뉴스 최신 기사 3개 찾아줘.",
    "코딩할 때 쓸만한 노트북 추천해줘.",
]
for question in sample_questions:
    print(f'질문 : {question} | 라우트 : {route_qeustion(question)}')    

질문 : 3개 상품을 2개씩 주문하면 총 몇 개인가? | 라우트 : calculator_tool
질문 : 나는 코딩용 노트북을 좋아한다는 점을 기억해줘. | 라우트 : memory_tool
질문 : AI 에이전트 뉴스 최신 기사 3개 찾아줘. | 라우트 : news_tool
질문 : 코딩할 때 쓸만한 노트북 추천해줘. | 라우트 : llm_recommendation


## tool 활용

In [46]:
import json
import os
import re
from urllib.parse import quote
from urllib.request import Request,urlopen


def calcualtor_tool(text:str)->float:
    allowed_chars = set("0123456789+-*/(). ")
    if not set(text) <= allowed_chars: # text의 문자중에 허용되지 않은 문자가 있다면
        raise ValueError('허용되지 않은 문자가 포함되어 있습니다.')
    return eval(text)

In [47]:
# 네이버 검색 API 예제 - 블로그 검색
import os
import re
import sys
import json
import html
import urllib.request
from datetime import datetime
from dotenv import load_dotenv
load_dotenv(override=True)

def _format_date(pubdate):
    return datetime.strptime(pubdate, "%a, %d %b %Y %H:%M:%S %z").strftime("%Y-%m-%d")

def _format_str(text):
    return html.unescape(re.sub(r'<[^>]+>',"",text))

client_id = os.getenv('NAVER_CLIENT_ID')
client_secret = os.getenv('NAVER_CLIENT_SECETET')

items = []
def search_naver_news(query:str, display:int=3)->list[dict]:
    encText = urllib.parse.quote(query)
    encText+= f'&display={display}&sort=date'
    url = "https://openapi.naver.com/v1/search/news?query=" + encText # JSON 결과    
    request = urllib.request.Request(url)
    request.add_header("X-Naver-Client-Id",client_id)
    request.add_header("X-Naver-Client-Secret",client_secret)


    response = urllib.request.urlopen(request)
    rescode = response.getcode()
    if(rescode==200):
        response_body = response.read().decode('utf-8')
        result = json.loads(response_body)

        for row in result.get('items'):
            items.append({
                'title':_format_str(row.get('title')),
                'content':_format_str(row.get('description')),                
                'date':_format_date(row.get('pubDate')),
                'link':row.get('link')
            })         
    return items

In [48]:
print(calcualtor_tool('30*50'))
search_naver_news('AI 에이전트')

1500


[{'title': '[오후 뉴스브리핑] 미 연준 카시카리 총재 “중동發 인플레 재확산 시 ...',
  'content': '베이스 MCP는 OAuth 2.1 기반 인증을 적용했으며, AI 에이전트가 요청한 거래는 이용자가 직접 확인하거나 취소해야 한다. MCP 서버는 이용자의 개인키를 보유하거나 접근하지 않는다고 밝혔다. 스페인, 폴리마켓·칼시... ',
  'date': '2026-05-27',
  'link': 'https://www.tokenpost.kr/news/briefing/363525'},
 {'title': 'KB금융, 그룹 사이버보안센터 출범…AI 보안 대응 확대',
  'content': 'AI 에이전트와 RPA(로봇 프로세스 자동화)를 결합해 최신 보안 위협과 취약점 정보를 실시간으로 수집·분석하고, 이상 행위 탐지와 정보 유출 징후 파악 등을 자동화했다. 악성메일 대응 훈련에도 AI를 활용해 최신 피싱... ',
  'date': '2026-05-27',
  'link': 'https://www.econovill.com/news/articleView.html?idxno=740895'},
 {'title': '알리바바 클라우드, 에이전틱 AI 풀스택 생태계 공개',
  'content': 'AI 에이전트가 데이터베이스, 빅데이터, 보안 등 클라우드 리소스를 함수 호출처럼 자연스럽게 활용할 수 있도록 설계됐으며, 경량 실행 샌드박스·태스크 간 메모리 공유·지능형 운영관리 등 인프라... ',
  'date': '2026-05-27',
  'link': 'https://platum.kr/archives/287675'}]

### 메모리 활용하기
- 이전대화나 사용자의 선호를 저장해서 다음 응답에 반영하는 기능

In [49]:
session_memory={}
def remember_preference(user_id:str, key:str, value:str)->None:
    if user_id not in session_memory:
        session_memory[user_id] = {}
    session_memory[user_id][key] = value

def get_preference(user_id:str, key:str, default:str | None = None) -> str | None:
    return session_memory.get(user_id,{}).get(key,default)

user_id = 'student-001'
remember_preference(user_id, 'category','노트북')
remember_preference(user_id, 'usage','코딩')

print(session_memory)

{'student-001': {'category': '노트북', 'usage': '코딩'}}


### 라우터 + 도구 + 메모리 통합
- 라우팅 규칙, 계산 도구, 네이버뉴스도구, 메모리 저장을 하나의 흐름으로 연결 --> Agent의 기본 형태
- 먼저 질문을 분류하고 그 분류 결과에 맞는 도구를 호출한뒤 필요하면 메모리까지 갱신
- 최종 결과를 llm에 전달해서 답변을 생성

In [50]:
def extract_math_expression(question: str) -> str:
    match = re.search(r"[0-9\s\+\-\*\/\(\)\.]+", question)
    if not match:
        raise ValueError("계산식을 찾을 수 없습니다.")
    expression = match.group(0).strip()
    if not expression:
        raise ValueError("계산식이 비어 있습니다.")
    return expression

def extract_news_query(question: str) -> str:
    cleaned = re.sub(r"뉴스|기사|검색|알려줘|찾아줘|추천해줘|좀|최근|최신|오늘", " ", question)
    cleaned = re.sub(r"\d+\s*개?", " ", cleaned)
    cleaned = re.sub(r"[^0-9A-Za-z가-힣\s]", " ", cleaned)
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    return cleaned or question

def route_and_execute(question: str) -> dict:
    route = route_qeustion(question)

    if route == "news_tool":
        news_query = extract_news_query(question)
        news_result = search_naver_news(news_query, display=3)
        news_result = recommend_product('뉴스 요약해줘',news_result)
        return {
            "route": route,
            "tool": "search_naver_news",
            "input": news_query,
            "result": news_result,
        }

    if route == "calculator_tool":
        expression = extract_math_expression(question)
        result = calcualtor_tool(expression)
        return {
            "route": route,
            "tool": "calculator_tool",
            "input": expression,
            "result": result,
        }

    if route == "memory_tool":
        remember_preference("student-001", "last_question", question)
        return {
            "route": route,
            "tool": "memory_tool",
            "input": question,
            "result": get_preference("student-001", "last_question"),
        }

    if route == "llm_recommendation":
        recommendation = recommend_product(question, context_string)
        return {
            "route": route,
            "tool": "llm_recommendation",
            "input": question,
            "result": recommendation,
        }

    return {
        "route": route,
        "tool": "general_llm",
        "input": question,
        "result": question,
    }

In [51]:
demo_questions = [
    # '3 + 2 * 4는 얼마야',
    '어제 오늘 사고뉴스 5개 찾아줘',
    '점심메뉴 추천해줘',
    '연구용 노트북 추천해줘',
]
print(json.loads(route_and_execute('어제 오늘 사고뉴스 5개 찾아줘')['result'])['assistant'])
print(json.loads(route_and_execute('어제 오늘 사고뉴스 5개 찾아줘')['result'])['reason'])
# for question in demo_questions:
#     result = route_and_execute(question)        
#     print(f'\n질문:{question}')
#     print(f'라우트:{result["route"]}')
#     print(f'도구:{result["tool"]}')
#     print(f'결과:{result["result"]}')
#     print(f'결과:{result["result"]["reason"]}')

오늘(2026-05-27) 주요 뉴스는 다음과 같습니다.

1) 서소문 고가차도 붕괴 사고
- 서울 서소문 고가차도 붕괴 사고 현장에서 경찰이 정밀 감식을 진행 중입니다.
- 복구 작업이 길어지면서 수도권 전철과 KTX 등 열차 운행에 차질이 이어지고 있습니다.
- 현장에서는 사고 원인 파악을 위한 추가 분석이 진행될 것으로 보입니다.

2) 철도 운행·예매 혼선으로 인한 승객 불편
- 사고 여파로 철도 운행이 줄거나 취소되면서 수원역 등 승객들이 불편을 겪고 있습니다.
- 일부 승객은 인터넷 예매가 원활하지 않아 직접 방문했지만 표를 구하기 어려웠다는 사례가 전해졌습니다.

3) 금융·보안/AI 관련 이슈
- KB금융이 그룹 사이버보안센터를 출범하고, AI 에이전트와 RPA를 결합해 보안 위협·취약점 정보를 실시간 수집·분석하며 이상 행위 탐지와 유출 징후 파악을 자동화하는 등 AI 보안 대응을 확대합니다.

4) 클라우드·AI 에이전트 생태계
- 알리바바 클라우드가 에이전틱 AI 풀스택 생태계를 공개하며, AI 에이전트가 데이터베이스·빅데이터·보안 등 클라우드 리소스를 함수 호출처럼 활용할 수 있도록 설계했다고 전했습니다.

5) 기술/인증 관련 소식(뉴스 브리핑)
- 베이스 MCP는 OAuth 2.1 기반 인증을 적용하고, AI 에이전트가 요청한 거래는 이용자가 직접 확인하거나 취소해야 하며, MCP 서버는 이용자의 개인키를 보유·접근하지 않는다고 안내했습니다.
제공된 컨텍스트의 기사들을 핵심 주제(서소문 고가 붕괴, 안전공학 코멘트, 금융/클라우드/보안 기술 이슈)로 묶어 간단히 요약했기 때문입니다.


In [52]:
# 1. 뉴스검색의 경우... 출력을 조정
# 2. 추천의 경우, 프롬프트가 현재 이커머스로 되어있는데-> 일반적인 프롬프트로 변경
